## Objective 
To apply and compare different feature selection techniques and evaluate 
their impact on model performance. 
### Task 
Using the Pima Indians Diabetes dataset, perform feature selection 
using the following techniques discussed in the lecture material 
(download the dataset from Kaggle) 
Instructions 
1. Dataset Preparation 
Load the dataset into Python. 
Separate the dataset into features XXX and target variable yyy. 
Perform any necessary preprocessing (e.g., handling missing values, 
feature scaling where required). 
2. Univariate Feature Selection 
Apply SelectKBest with the chi-squared (chi²) test. 
Select the top 4 features. 
Display the selected features and their scores. 
3. Recursive Feature Elimination (RFE) 
Use Logistic Regression as the base estimator. 
© Copyright 2025 MIVA Open University All Rights Reserved | 2 
Apply RFE to select the top 3 features. 
List the selected features. 
4. Principal Component Analysis (PCA) 
Apply PCA to reduce the dataset to 3 principal components. 
Report the explained variance ratio of each component. 
5. Feature Importance 
Train an ExtraTreesClassifier on the dataset. 
Compute and rank feature importance scores. 
Identify the top 3 most important features. 
6. Model Evaluation 
Train a Logistic Regression classifier using: 
- (a) All original features 
- (b) Features selected using Univariate Selection 
- Features selected using RFE 
- Compare classification accuracy for each case. 

Submission Requirements 
Python code (well-commented). 
Tables or printed outputs showing selected features and scores. 
© Copyright 2025 MIVA Open University All Rights Reserved | 3.

A brief discussion (5–7 sentences) comparing the results of the 
different feature selection techniques and their effect on model 
performance. 
Assessment Focus 
Correct application of feature selection methods 
Interpretation of outputs 
Clarity of results and discussion

In [1]:
# import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import sklearn
import os 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE

In [2]:
# import the Pima Indiansdata from kaggle
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")

print("Path to dataset files:", path)


c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\uciml\pima-indians-diabetes-database\versions\1


In [3]:
# Load the dataset
df = pd.read_csv(os.path.join(path, 'diabetes.csv'))

# Display the first few rows to verify
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# separate the features and target variable
X = df.drop('Outcome', axis=1)  # Features
y = df['Outcome']  # Target variable


In [5]:
# Check for missing values
print(X.isnull().sum())

#

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [6]:
# Handle missing values (replace 0s in certain columns with NaN and impute)
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X[cols_with_zeros] = X[cols_with_zeros].replace(0, np.nan)
X.fillna(X.mean(), inplace=True)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.00000,180.000000,32.9,0.171,63
764,2,122.0,70.0,27.00000,155.548223,36.8,0.340,27
765,5,121.0,72.0,23.00000,112.000000,26.2,0.245,30
766,1,126.0,60.0,29.15342,155.548223,30.1,0.349,47


In [7]:
# Check for missing values
print(X.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [8]:
# feature scaling

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  


In [9]:
X_scaled

array([[ 0.63994726,  0.86510807, -0.03351824, ...,  0.16629174,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.20616153, -0.52985903, ..., -0.85253118,
        -0.36506078, -0.19067191],
       [ 1.23388019,  2.0158134 , -0.69530596, ..., -1.33283341,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 , -0.0225789 , -0.03351824, ..., -0.91074963,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.14180757, -1.02619983, ..., -0.34311972,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.94314317, -0.19896517, ..., -0.29945588,
        -0.47378505, -0.87137393]], shape=(768, 8))

In [10]:
# display for where X values are less than 0
print(X_scaled[X_scaled < 0])

[-3.35182392e-02 -3.34507888e-16 -8.44885053e-01 ... -2.99455878e-01
 -4.73785050e-01 -8.71373930e-01]


In [11]:
# Split data to prevent data leakage during scaling and selection

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Original features: {X.columns.tolist()}\n")

# --- APPROACH A: Using chi2 (Requires non-negative data) ---
# We use MinMaxScaler to scale data into the [0, 1] range.
pipeline_chi2 = Pipeline([
    ('scaler', MinMaxScaler()),
    ('selector', SelectKBest(chi2, k=3)), # Select top 4 features
    ('classifier', LogisticRegression(solver='liblinear'))
])

pipeline_chi2.fit(X_train, y_train)
print(f"Chi-Square Pipeline Accuracy: {pipeline_chi2.score(X_test, y_test):.4f}")

# Get the features selected by the chi2 pipeline
mask_chi2 = pipeline_chi2.named_steps['selector'].get_support()
selected_feats_chi2 = X.columns[mask_chi2]
print(f"Features selected by Chi2: {selected_feats_chi2.tolist()}\n")

# display the features with their chi2 scores
chi2_scores = pipeline_chi2.named_steps['selector'].scores_
for feat, score in zip(X.columns, chi2_scores):
    print(f"{feat}: {score:.4f}")


Original features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Chi-Square Pipeline Accuracy: 0.7229
Features selected by Chi2: ['Pregnancies', 'Glucose', 'Age']

Pregnancies: 4.8702
Glucose: 13.0520
BloodPressure: 0.5036
SkinThickness: 1.2772
Insulin: 2.1836
BMI: 3.4428
DiabetesPedigreeFunction: 1.4147
Age: 5.2312


In [12]:
# --- APPROACH B: Using RFE with Logistic Regression ---

model = LogisticRegression()
rfe = RFE(model,n_features_to_select=3)
X_rfe = rfe.fit_transform(X,y)


c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
# --- APPROACH B: Using Recursive Feature Elimination (RFE) with Logistic Regression ---

columns = df.columns.tolist()
pipeline_rfe = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', RFE(estimator=LogisticRegression(solver='liblinear'), n_features_to_select=3)),
    ('classifier', LogisticRegression(solver='liblinear'))
])
pipeline_rfe.fit(X_train, y_train)
print(f"RFE Pipeline Accuracy: {pipeline_rfe.score(X_test, y_test):.3f}")

# select the top 3 features using RFE
mask_rfe = pipeline_rfe.named_steps['selector'].get_support()
selected_feats_rfe = X.columns[mask_rfe]
print(f"Features selected by RFE: {selected_feats_rfe.tolist()}\n") 

    
    

RFE Pipeline Accuracy: 0.745
Features selected by RFE: ['Pregnancies', 'Glucose', 'BMI']



In [14]:
# using Principal Component Analysis (PCA) for dimensionality reduction 

n_components = 3
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

loadings = pca.components_

feature_names = [f'Feature_{i+1}' for i in range(X_scaled.shape[1])]

# 4. Create the DataFrame
pca_df = pd.DataFrame(
    loadings, 
    columns=feature_names, 
    index=[f'PC{i+1}' for i in range(n_components)]
)

print(pca_df.T.head(3))



                PC1       PC2       PC3
Feature_1  0.308371  0.552069 -0.092072
Feature_2  0.421065 -0.068170  0.455478
Feature_3  0.378498  0.139202 -0.317728


In [15]:
#Train an ExtraTreesClassifier on the dataset


model = ExtraTreesClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"ExtraTreesClassifier Accuracy: {model.score(X_test, y_test):.3f}")

# Compute and rank feature importance scores.
feature_importance = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)
print("\nFeature Importance Scores:")
print(importance_df)

ExtraTreesClassifier Accuracy: 0.740

Feature Importance Scores:
                    feature  importance
1                   Glucose    0.246230
5                       BMI    0.139498
7                       Age    0.131383
6  DiabetesPedigreeFunction    0.112617
0               Pregnancies    0.103268
2             BloodPressure    0.093153
4                   Insulin    0.089094
3             SkinThickness    0.084757


In [16]:
# Identify the top 3 most important features. 
top_3_features = importance_df.head(3)['feature'].tolist()
print(f"Top 3 Important Features: {top_3_features}")

Top 3 Important Features: ['Glucose', 'BMI', 'Age']


## Model Evaluation

In [17]:
 
#Train a Logistic Regression classifier using: All original features 
pipeline_all = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(solver='liblinear'))
])
pipeline_all.fit(X_train, y_train)
print(f"Logistic Regression Accuracy (All Features): {pipeline_all.score(X_test, y_test):.3f}")


Logistic Regression Accuracy (All Features): 0.740


In [18]:
#Train a Logistic Regression classifier using: Features selected using Univariate Selection
pipeline_univariate = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(solver='liblinear'))
])
pipeline_univariate.fit(X_train, y_train)
print(f"Logistic Regression Accuracy (Univariate Selection): {pipeline_univariate.score(X_test, y_test):.3f}")



Logistic Regression Accuracy (Univariate Selection): 0.740


In [19]:
#Train a Logistic Regression classifier using: Features selected using RFE 
pipeline_rfe = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', RFE(estimator=LogisticRegression(solver='liblinear'), n_features_to_select=3)),
    ('classifier', LogisticRegression(solver='liblinear'))
])
pipeline_rfe.fit(X_train, y_train)
print(f"RFE Pipeline Accuracy: {pipeline_rfe.score(X_test, y_test):.3f}")   



RFE Pipeline Accuracy: 0.745


In [20]:
# Compare classification accuracy for each case. 
print(f"Logistic Regression Accuracy (All Features): {pipeline_all.score(X_test, y_test):.3f}") 
print(f"Logistic Regression Accuracy (Univariate Selection): {pipeline_univariate.score(X_test, y_test):.3f}")
print(f"RFE Pipeline Accuracy: {pipeline_rfe.score(X_test, y_test):.3f}")   

Logistic Regression Accuracy (All Features): 0.740
Logistic Regression Accuracy (Univariate Selection): 0.740
RFE Pipeline Accuracy: 0.745
